# Transformixer mixer ablation

Controlled ablation under a shared **RevIN + NLinear** temporal front-end.

| ID | Variant | Mixer |
|----|---------|-------|
| A | `xlstm_full` | sLSTM + memory tokens + forward/reverse (xLSTM-Mixer FULL) |
| B | `tf_no_pe` | Transformixer **without** positional encodings (paper default) |
| C | `tf_with_pe` | Transformixer **with** sinusoidal PE on variate tokens |
| D | `nlinear_only` | Shared NLinear only (no variate mixer) |


In [ ]:
import os
import sys
from pathlib import Path


# Run from the transformixer project root so dataset/checkpoint paths resolve correctly.
TRANSFORMIXER_ROOT = (Path.cwd().parent / "models" / "transformixer").resolve()
os.chdir(TRANSFORMIXER_ROOT)
sys.path.insert(0, str(TRANSFORMIXER_ROOT))

In [ ]:
# Shared training recipe (matches paper notebooks).
seq_len = 96
pred_len = 96
lr = 5e-4
batch_size = 32
d_model = 1024
e_layers = 2
n_heads = 16
dropout = 0.1
gamma = 0.99
cosine_epochs = 5
warmup_epochs = 2
constant_gamma_epochs = 1
seed = 42
max_epochs = 10
fast_dev_run = False  # set True for a 1-batch smoke test
num_workers = 4

# Datasets to evaluate.
DATASETS = ["Electricity", "Traffic"]

# Which variants to run (comment out to skip).
# A: xLSTM-Mixer FULL, B: Transformixer no PE, C: Transformixer + PE, D: NLinear only
VARIANTS = ["xlstm_full", "tf_no_pe", "tf_with_pe", "nlinear_only"]

# xLSTM-Mixer FULL hyperparameters (same as notebooks/xlstm-mixer*.ipynb).
xlstm_kwargs = dict(
    num_mem_tokens=3,
    xlstm_embedding_dim=d_model,
    xlstm_num_blocks=e_layers,
    xlstm_num_heads=n_heads,
    xlstm_conv1d_kernel_size=4,
    xlstm_dropout=dropout,
)

root_path = "../../datasets"
results_csv = Path(TRANSFORMIXER_ROOT) / "outputs" / "mixer_ablation_results.csv"
results_csv.parent.mkdir(parents=True, exist_ok=True)

print("Datasets:", DATASETS)
print("Variants:", VARIANTS)
print("pred_len / max_epochs / seed:", pred_len, max_epochs, seed)
print("Results will append to:", results_csv)


In [ ]:
import gc
from typing import Any

import torch
import wandb
from lightning.pytorch.callbacks import StochasticWeightAveraging

from xlstm_mixer.cli_helper import LoggerSaveConfigCallback, TaskCLI
from xlstm_mixer.exp.exp import ForecastingExp
from xlstm_mixer.lit.data import TSLibDataModule


def _extract_metrics(logged: dict) -> dict[str, float]:
    if "test/MeanSquaredError" in logged:
        return {
            "mse": float(logged["test/MeanSquaredError"].item()),
            "mae": float(logged["test/MeanAbsoluteError"].item()),
        }
    return {
        "mae": float(logged["test/MeanAbsoluteError"].item()),
        "mape": float(logged["test/MeanAbsolutePercentageError"].item()),
        "rmse": float(logged["test/RootMeanSquaredError"].item()),
    }


def build_rest_args(dataset: str, variant: str) -> list[str]:
    common = [
        "--data", "ForecastingTSLibDataModule",
        "--data.dataset_name", dataset,
        "--data.root_path", root_path,
        "--optimizer.lr", str(lr),
        "--data.seq_len", str(seq_len),
        "--data.pred_len", str(pred_len),
        "--data.label_len", "0",
        "--data.batch_size", str(batch_size),
        "--data.num_workers", str(num_workers),
        "--data.persistent_workers", "true" if num_workers > 0 else "false",
        "--model", "LongTermForecastingExp",
        "--model.criterion", "torch.nn.L1Loss",
        "--lr_scheduler.constant_gamma_epochs", str(constant_gamma_epochs),
        "--lr_scheduler.gamma", str(gamma),
        "--lr_scheduler.cosine_epochs", str(cosine_epochs),
        "--lr_scheduler.warmup_epochs", str(warmup_epochs),
        "--trainer.logger.name", f"{dataset}_{variant}_{pred_len}_{seed}",
        "--trainer.logger.project", "transformixer-ablation",
        "--trainer.logger.mode", "offline",
        "--trainer.max_epochs", str(max_epochs),
        "--seed_everything", str(seed),
        "--trainer.fast_dev_run", str(fast_dev_run).lower(),
    ]

    if variant == "xlstm_full":
        arch = [
            "--model.architecture", "xLSTMMixer",
            "--model.architecture.num_mem_tokens", str(xlstm_kwargs["num_mem_tokens"]),
            "--model.architecture.xlstm_num_heads", str(xlstm_kwargs["xlstm_num_heads"]),
            "--model.architecture.xlstm_num_blocks", str(xlstm_kwargs["xlstm_num_blocks"]),
            "--model.architecture.xlstm_embedding_dim", str(xlstm_kwargs["xlstm_embedding_dim"]),
            "--model.architecture.xlstm_conv1d_kernel_size", str(xlstm_kwargs["xlstm_conv1d_kernel_size"]),
            "--model.architecture.xlstm_dropout", str(xlstm_kwargs["xlstm_dropout"]),
            # FULL is the default AblationMode; memory + backcast enabled via defaults + num_mem_tokens.
        ]
    elif variant == "tf_no_pe":
        arch = [
            "--model.architecture", "Transformixer",
            "--model.architecture.n_heads", str(n_heads),
            "--model.architecture.e_layers", str(e_layers),
            "--model.architecture.d_model", str(d_model),
            "--model.architecture.dropout", str(dropout),
            "--model.architecture.use_positional_encoding", "false",
            "--model.architecture.use_variate_mixer", "true",
        ]
    elif variant == "tf_with_pe":
        arch = [
            "--model.architecture", "Transformixer",
            "--model.architecture.n_heads", str(n_heads),
            "--model.architecture.e_layers", str(e_layers),
            "--model.architecture.d_model", str(d_model),
            "--model.architecture.dropout", str(dropout),
            "--model.architecture.use_positional_encoding", "true",
            "--model.architecture.use_variate_mixer", "true",
        ]
    elif variant == "nlinear_only":
        arch = [
            "--model.architecture", "Transformixer",
            "--model.architecture.use_variate_mixer", "false",
            # unused when mixer is off, but CLI still accepts defaults
            "--model.architecture.n_heads", str(n_heads),
            "--model.architecture.e_layers", str(e_layers),
            "--model.architecture.d_model", str(d_model),
            "--model.architecture.dropout", str(dropout),
        ]
    else:
        raise ValueError(f"Unknown variant: {variant}")

    return common + arch


def run_one(dataset: str, variant: str) -> dict[str, Any]:
    print("=" * 72)
    print(f"Running {variant} on {dataset} (H={pred_len}, seed={seed})")
    print("=" * 72)

    rest_args = build_rest_args(dataset, variant)
    cli = TaskCLI(
        ForecastingExp,
        TSLibDataModule,
        subclass_mode_data=True,
        subclass_mode_model=True,
        run=False,
        args=rest_args,
        save_config_callback=LoggerSaveConfigCallback,
    )
    cli.datamodule.root_path = Path(root_path)

    # Sanity: print mixer flags when Transformixer.
    arch = cli.model.model
    print("Architecture:", type(arch).__name__)
    for attr in ("use_positional_encoding", "use_variate_mixer", "num_mem_tokens", "backcast", "ablation_mode"):
        if hasattr(arch, attr):
            print(f"  {attr} =", getattr(arch, attr))

    cli.trainer.fit(cli.model, cli.datamodule)
    cli.trainer.callbacks = [
        cb for cb in cli.trainer.callbacks if not isinstance(cb, StochasticWeightAveraging)
    ]

    row: dict[str, Any] = {
        "dataset": dataset,
        "variant": variant,
        "pred_len": pred_len,
        "seq_len": seq_len,
        "seed": seed,
        "max_epochs": max_epochs,
    }

    if cli.trainer.fast_dev_run:
        print("Fast dev run — skipping test.")
        row["status"] = "fast_dev_run"
        row.update({k: None for k in ("mse", "mae")})
    else:
        cli.trainer.test(
            cli.model,
            cli.datamodule,
            ckpt_path=cli.trainer.checkpoint_callback.best_model_path,
        )
        metrics = _extract_metrics(cli.trainer.logged_metrics)
        row.update(metrics)
        row["status"] = "ok"
        print("Metrics:", metrics)

    try:
        wandb.finish()
    except Exception:
        pass

    # Free GPU memory between runs.
    del cli
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row


In [ ]:
import pandas as pd

rows = []
# Resume-friendly: skip combos already present in CSV with status=ok.
done = set()
if results_csv.exists():
    prev = pd.read_csv(results_csv)
    for _, r in prev.iterrows():
        if r.get("status") == "ok":
            done.add((r["dataset"], r["variant"], int(r["pred_len"]), int(r["seed"])))
    print(f"Loaded {len(done)} finished runs from {results_csv}")

for dataset in DATASETS:
    for variant in VARIANTS:
        key = (dataset, variant, pred_len, seed)
        if key in done:
            print(f"SKIP (already done): {key}")
            continue
        row = run_one(dataset, variant)
        rows.append(row)
        # Append immediately so interrupted Colab sessions keep partial results.
        df_new = pd.DataFrame([row])
        write_header = not results_csv.exists()
        df_new.to_csv(results_csv, mode="a", header=write_header, index=False)
        print(f"Appended to {results_csv}")

print("Finished scheduled runs:", len(rows))


In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(results_csv)
# Keep latest ok row per (dataset, variant, pred_len, seed).
df_ok = df[df["status"] == "ok"].copy()
df_ok = df_ok.drop_duplicates(subset=["dataset", "variant", "pred_len", "seed"], keep="last")

variant_order = ["xlstm_full", "tf_no_pe", "tf_with_pe", "nlinear_only"]
df_ok["variant"] = pd.Categorical(df_ok["variant"], categories=variant_order, ordered=True)
df_ok = df_ok.sort_values(["dataset", "variant"])

print("Full ablation results")
display(df_ok)

# Paper-style pivot: MSE / MAE by dataset × variant
if {"mse", "mae"}.issubset(df_ok.columns):
    pivot_mse = df_ok.pivot_table(index="variant", columns="dataset", values="mse")
    pivot_mae = df_ok.pivot_table(index="variant", columns="dataset", values="mae")
    print("\nMSE")
    display(pivot_mse)
    print("\nMAE")
    display(pivot_mae)

    # Relative to xLSTM FULL (negative = Transformixer better)
    if "xlstm_full" in pivot_mse.index:
        print("\nRelative MSE vs xlstm_full (negative is better)")
        display((pivot_mse / pivot_mse.loc["xlstm_full"] - 1.0) * 100.0)

# Optional LaTeX snippet for the paper
try:
    print("\nLaTeX (MSE):")
    print(pivot_mse.to_latex(float_format="%.4f"))
except Exception:
    pass


## How to read the table

- **B vs C (`tf_no_pe` vs `tf_with_pe`)**: tests the PE-free / set-equivariance claim.
- **B vs A (`tf_no_pe` vs `xlstm_full`)**: tests Transformer mixer vs ordered sLSTM under the same RevIN+NLinear front-end and training recipe.
- **D (`nlinear_only`)**: temporal baseline; shows how much the mixer helps.